In [ ]:
from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from import_folder import import_all_noiselevel_data
from sklearn.model_selection import train_test_split

# -----------------------------------------------------------------------------
# Data Load and Definitions
# -----------------------------------------------------------------------------
folders = ['SNR_noiseless', 'SNR_50dB', 'SNR_40dB', 'SNR_30dB', 'SNR_20dB']

# Load all noise levels into a single DataFrame
all_data = import_all_noiselevel_data(folders, add_noise_column=False)

# Separate Features and Label
X = all_data.drop(columns=['label'])
y = all_data['label']
# Define cv strategy for hyperparameter tuning
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define scoring metric for hyperparameter tuning
scoring_metric = 'f1_macro'

# random search parameters
num_combinations = 20  # Number of hyperparameter sets sampled

# other definitions
RANDOM_STATE = 42

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
# -----------------------------------------------------------------------------
# Multilayer Perceptron (MLP) Randomized Search
# -----------------------------------------------------------------------------
mlp_param_dist = {
    "classifier__hidden_layer_sizes": [
        (128, 64, 32),
        (256, 128, 64),
        (512, 256, 128),
        (128, 64),
        (64, 64, 32),
        (256, 128),
        (512, 256),
        (128,),
        (64,),
    ],
    "classifier__activation": ["relu", "tanh"],
    "classifier__alpha": loguniform(1e-5, 1e-2),
    "classifier__max_iter": randint(200, 1000),
}

mlp_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),  # Crucial for neural network convergence
        (
            "classifier",
            MLPClassifier(
                max_iter=1000, random_state=RANDOM_STATE, early_stopping=True
            ),
        ),
    ]
)

mlp_random = RandomizedSearchCV(
    estimator=mlp_pipeline,
    param_distributions=mlp_param_dist,
    n_iter=num_combinations,
    scoring=scoring_metric,
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# -----------------------------------------------------------------------------
# Fit the Randomized Searches on Training Data
# -----------------------------------------------------------------------------
print("Fitting MLP Neural Network Randomized Search...")
mlp_random.fit(X_train, y_train)

print("\n=== TUNING COMPLETE ===")
print(f"Best MLP CV {scoring_metric}: {mlp_random.best_score_:.4f}")

# -----------------------------------------------------------------------------
# Inspect Best Hyperparameters Found by Randomized Search
# -----------------------------------------------------------------------------
print("=== BEST HYPERPARAMETERS FOUND ===")

print("\n3. Multilayer Perceptron (MLP):")
for param, value in mlp_random.best_params_.items():
    print(f"   {param}: {value}")

=== INITIALIZING TASK 2 RANDOMIZED SEARCH PIPELINES ===

Fitting Decision Tree Randomized Search...
Fitting Random Forest Randomized Search...
Fitting MLP Neural Network Randomized Search...

=== TUNING COMPLETE ===
Best DT CV f1_macro: 0.9777
Best RF CV f1_macro: 0.9851
Best MLP CV f1_macro: 0.9807
=== BEST HYPERPARAMETERS FOUND ===

1. Decision Tree (DT):
   classifier__min_samples_split: 5
   classifier__min_samples_leaf: 5
   classifier__max_depth: 16
   classifier__criterion: entropy

2. Random Forest (RF):
   classifier__n_estimators: 500
   classifier__min_samples_split: 2
   classifier__max_features: sqrt
   classifier__max_depth: None

3. Multilayer Perceptron (MLP):
   classifier__learning_rate_init: 0.01
   classifier__hidden_layer_sizes: (128, 64)
   classifier__alpha: 1e-05
   classifier__activation: relu
